# Messi vs CR7: ¿quién llega primero a 1000 goles?
## Fase 1: exploración y validación de datos

Objetivo de este notebook: **consolidar la cifra actual de goles oficiales de cada jugador** y entender cómo vienen las fechas en cada fuente, antes de construir el modelo.

| Archivo | Qué es | Granularidad |
|---|---|---|
| `Messi_vs_Ronaldo_Stats_By-NourHesham.xlsx` | Resúmenes de ambos (16 hojas) | Por año, temporada, edad, club... |
| `messi_goles.csv` | Todos los goles de Messi (club + Argentina) | 1 fila = 1 gol |
| `ronaldo_goals_kaggle_ready.csv` | Goles de CR7 **solo con clubes** | 1 fila = 1 gol |

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option('display.width', 200)

RUTA = Path(r"C:\Users\DANIEL\OneDrive\Escritorio\Messi vs Cristiano")

for f in sorted(RUTA.iterdir()):
    print(f"{f.name:<50} {f.stat().st_size/1024:>7.1f} KB")

.streamlit                                             4.0 KB
.vs                                                    0.0 KB
01_exploracion_datos.ipynb                            57.5 KB
02_modelo_series_temporales.ipynb                    141.5 KB
03_modelo_bayesiano.ipynb                            321.5 KB
app.py                                                15.2 KB
data_limpia                                            4.0 KB
messi_goles.csv                                       95.6 KB
Messi_vs_CR7_Goals.ipynb                               8.9 KB
Messi_vs_Ronaldo_Stats_By-NourHesham.xlsx             40.6 KB
README.md                                              1.2 KB
requirements.txt                                       0.0 KB
ronaldo_goals_kaggle_ready.csv                        99.4 KB


## 1. El Excel

`pd.ExcelFile` abre el archivo una sola vez; después se lee cualquier hoja sin volver a cargarlo desde disco.

In [2]:
xls = pd.ExcelFile(RUTA / "Messi_vs_Ronaldo_Stats_By-NourHesham.xlsx")

for i, h in enumerate(xls.sheet_names):
    print(i, h)

0 Overview
1 By Season
2 By Calendar Year
3 By Age
4 Club Stats
5 Competition Breakdown
6 International Stats
7 UCL & World Cup
8 Head-to-Head
9 El Clasico & Finals
10 Favourite Opponents
11 Trophies
12 Shooting, Dribbling & Passing
13 Who Is Better (Verdict)
14 Player Bios & Records
15 Sources & Notes


Las hojas **no empiezan en la fila 1**: arriba hay un título, un subtítulo y una fila vacía. Por eso, en tu primer intento, pandas te devolvió columnas `Unnamed: 1, Unnamed: 2...`.

Así se ve la hoja cruda (`header=None` = "no adivines el encabezado"):

In [3]:
crudo = pd.read_excel(xls, "By Calendar Year", header=None)
crudo.head(6)

,0,1,2,3
0,Goals by Calendar Year,NaN,NaN,NaN
1,All-competition goals scored in each calendar ...,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN
3,Year,Messi Goals,Ronaldo Goals,Difference (Messi − Ronaldo)
4,2002,0,5,-5
5,2003,—,1,—


El encabezado real está en la fila con índice **3** → `header=3`. Además:
- abajo hay una nota `Source: ...` que hay que quitar,
- los vacíos vienen como `—` (texto), así que la columna queda como `object` en vez de número. `pd.to_numeric(..., errors="coerce")` convierte lo que pueda y lo demás lo deja en `NaN`.

Como varias hojas tienen la misma estructura, lo encapsulamos en una función:

In [4]:
def leer_hoja(nombre, hdr=3):
    df = pd.read_excel(xls, nombre, header=hdr).dropna(how="all")
    col0 = df.columns[0]
    return df[~df[col0].astype(str).str.startswith(("Source", "Note"))]


anual = leer_hoja("By Calendar Year")
anual.columns = ["year","messi", "ronaldo", "diff"]
anual = anual.apply(pd.to_numeric, errors="coerce")
anual["year"] = anual["year"].astype(int)

anual.info()
anual.tail()

<class 'pandas.core.frame.DataFrame'>
Index: 25 entries, 0 to 24
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   year     25 non-null     int64  
 1   messi    24 non-null     float64
 2   ronaldo  25 non-null     float64
 3   diff     24 non-null     float64
dtypes: float64(3), int64(1)
memory usage: 1000.0 bytes


,year,messi,ronaldo,diff
20,2022,35.0,16.0,19.0
21,2023,28.0,54.0,-26.0
22,2024,29.0,43.0,-14.0
23,2025,46.0,41.0,5.0
24,2026,26.0,20.0,6.0


### Cuatro hojas, cuatro formas de contar el total

Si los datos son consistentes, el total de goles debería ser el mismo sin importar cómo lo sumes.

In [5]:
ov = leer_hoja("Overview").set_index("Metric")
ov.loc[["Total Goals", "Total Appearances", "Goals per Game"]]

,Lionel Messi,Cristiano Ronaldo,Difference (Messi − Ronaldo)
Metric,,,
Total Goals,922,977,-55
Total Appearances,1168,1332,-164
Goals per Game,0.79,0.73,0.06


In [6]:
temporadas = leer_hoja("By Season")
temporadas.tail(4)

,Season,Messi Apps,Messi Goals,Messi Assists,Messi G/Game,Ronaldo Apps,Ronaldo Goals,Ronaldo Assists,Ronaldo G/Game
22,2024/25,46,36,14,0.78,50.0,43.0,5.0,0.86
23,2025/26,49,45,30,0.92,49.0,38.0,4.0,0.78
24,2026/27,4,3,2,0.75,2.0,1.0,0.0,0.50
25,Career Total,1168,922,424,0.79,1332.0,977.0,260.0,0.73


In [7]:
edad = leer_hoja("By Age")
edad = edad.apply(pd.to_numeric, errors="coerce")
edad.tail(4)

,Age,Messi Goals (age),Messi Cumulative,Ronaldo Goals (age),Ronaldo Cumulative
21,38,50.0,916.0,53.0,873.0
22,39,3.0,919.0,50.0,923.0
23,40,NaN,NaN,38.0,961.0
24,41,NaN,NaN,15.0,976.0


In [8]:
temp_ok = temporadas[temporadas["Season"] != "Career Total"]

chequeo = pd.DataFrame({
    "Overview": [ov.loc["Total Goals", "Lionel Messi"], ov.loc["Total Goals", "Cristiano Ronaldo"]],
    "Σ temporadas": [pd.to_numeric(temp_ok["Messi Goals"], errors="coerce").sum(),
                     pd.to_numeric(temp_ok["Ronaldo Goals"], errors="coerce").sum()],
    "Σ años calendario": [anual.messi.sum(), anual.ronaldo.sum()],
    "Acumulado por edad": [edad["Messi Cumulative"].max(), edad["Ronaldo Cumulative"].max()],
}, index=["Messi", "Ronaldo"]).astype(int)

chequeo

,Overview,Σ temporadas,Σ años calendario,Acumulado por edad
Messi,922,922,922,919
Ronaldo,977,977,977,976


Overview, temporadas y años calendario cuadran (**922** y **977**). La hoja **By Age** se queda corta (919 / 976): se actualizó antes que las demás, así que **no la usaremos como fuente de totales** (sí servirá luego para la ponderación por etapa de carrera).

### Club vs selección

El CSV de Ronaldo solo trae goles de club, así que necesitamos el desglose del Excel para poder cruzar.

In [9]:
clubs = pd.read_excel(xls, "Club Stats", header=None, usecols=[0, 2]).dropna()
clubs.columns = ["club", "goles"]
clubs = clubs[clubs.goles != "Goals"].set_index("club")
clubs["goles"] = clubs.goles.astype(int)

intl = leer_hoja("International Stats").set_index("Metric")

print(clubs, "\n")
print("Goles con selección ->",
      "Messi:", intl.loc["Goals", "Messi (Argentina)"],
      "| Ronaldo:", intl.loc["Goals", "Ronaldo (Portugal)"])

                     goles
club                      
FC Barcelona           672
Paris Saint-Germain     32
Inter Miami             93
Messi Total            797
Sporting CP              5
Manchester United      145
Real Madrid            450
Juventus               101
Al-Nassr               130
Ronaldo Total          831 

Goles con selección -> Messi: 125 | Ronaldo: 146


Messi: 797 club + 125 Argentina = 922 ✔️ · Ronaldo: 831 club + 146 Portugal = 977 ✔️

## 2. CSV de Messi (gol a gol)

Cosas a notar en el formato:
- `Fecha` viene como `20-9-2026` y también `22-01-2006` (con y sin cero). El formato `%d-%m-%Y` acepta ambos.
- `Minuto` es texto porque incluye tiempo añadido (`90+6`).
- `Competición` tiene códigos pegados al nombre: `PRMLigue 1`, `WCQWorld Cup 2022 Qualification`...

In [10]:
messi = pd.read_csv(RUTA / "messi_goles.csv")
print(messi.shape)
messi.head(3)

(930, 12)


,Índice,Fecha,Equipo Local,Resultado,Equipo Visitante,Marcador,Tipo,Cómo,Competición,Camiseta,Minuto,Video
0,930,20-9-2026,Inter Miami CF,2-2,San Diego FC,1-1,Tiro libre,Pie Izquierdo,Major League Soccer,10,24,https://www.youtube.com/watch?v=s34OfEUkPNs
1,929,16-9-2026,Inter Miami CF,2-0,Cruz Azul,1-0,Gol de campo,Cabeza,Campeones Cup,10,24,https://www.youtube.com/watch?v=dnKIRO0ht90
2,928,12-9-2026,Inter Miami CF,2-2,Nashville SC,2-1,Gol de campo,Pie Izquierdo,Major League Soccer,10,62,https://www.youtube.com/watch?v=sEIbHwaUXz8


In [11]:
messi.isna().sum()

Índice                0
Fecha                 0
Equipo Local          0
Resultado             0
Equipo Visitante      0
Marcador              0
Tipo                278
Cómo                  0
Competición           0
Camiseta              0
Minuto                0
Video               655
dtype: int64

Los nulos están en `Tipo` y `Video`, que no afectan el conteo. Ahora limpiamos y creamos columnas útiles:

In [12]:
messi["fecha"] = pd.to_datetime(messi["Fecha"], format="%d-%m-%Y")
messi["anio"] = messi.fecha.dt.year
messi["comp"] = messi["Competición"].str.replace(r"^(PRM|NAM|SUP|WCQ|WCP|WCT|CPA|IUP)", "", regex=True)
messi["seleccion"] = messi[["Equipo Local", "Equipo Visitante"]].eq("Argentina").any(axis=1)

print("Rango:", messi.fecha.min().date(), "->", messi.fecha.max().date())
print("Goles:", len(messi), "| Índice máx:", messi["Índice"].max(), "| filas duplicadas:", messi.duplicated().sum())
print(messi.seleccion.map({True: "Argentina", False: "Club"}).value_counts())

Rango: 2005-05-01 -> 2026-09-20
Goles: 930 | Índice máx: 930 | filas duplicadas: 0
seleccion
Club         805
Argentina    125
Name: count, dtype: int64


In [13]:
messi.comp.value_counts()

comp
La Liga                          474
UEFA Champions League            129
Major League Soccer               70
Copa del Rey                      56
Amistoso Internacional            54
Ligue 1                           22
Leagues Cup                       14
Spanish Super Cup                 14
World Cup 2014 Qualification      10
World Cup 2026                     8
Eliminatorias Mundial 2026         8
Concacaf Champions Cup             8
World Cup 2022                     7
Major League Soccer Play Offs      7
World Cup 2018 Qualification       7
World Cup 2022 Qualification       7
FIFA Club World Cup                6
Copa América Centenario            5
World Cup 2010 Qualification       4
World Cup 2014                     4
Copa América 2021                  4
UEFA European Super Cup            3
Copa América 2007                  2
Trophée des Champions              1
Campeones Cup                      1
Copa América 2024                  1
World Cup 2018                   

## 3. CSV de Ronaldo (solo clubes)

- `encoding="utf-8-sig"`: el archivo trae un carácter invisible (BOM) al inicio. Sin esto, la primera columna se llamaría `\ufeffcompetition`.
- `match_date` viene como `07/10/02` → día/mes/**año de 2 dígitos** → `%d/%m/%y`.
- `competition` trae espacios al final (`"Liga Portugal "`), así que hay que hacer `strip()`.

In [14]:
cr7 = pd.read_csv(RUTA / "ronaldo_goals_kaggle_ready.csv", encoding="utf-8-sig")
cr7["fecha"] = pd.to_datetime(cr7.match_date, format="%d/%m/%y")
cr7["anio"]  = cr7.fecha.dt.year
cr7["competition"] = cr7.competition.str.strip()

print(cr7.shape, "|", cr7.fecha.min().date(), "->", cr7.fecha.max().date())
cr7.competition.value_counts()

(816, 18) | 2002-10-07 -> 2026-02-25


competition
LaLiga                   311
UEFA Champions League    140
Premier League           103
Saudi Pro League          95
Serie A                   81
Copa del Rey              22
AFC Champions League      14
FA Cup                    13
Club World Cup             7
Italy Cup                  4
EFL Cup                    4
Supercopa                  4
King's Cup                 3
Liga Portugal              3
Saudi Super Cup            3
Taça de Portugal           2
Europa League              2
UEFA Super Cup             2
Supercoppa Italiana        2
Champions League Qu.       1
Name: count, dtype: int64

In [15]:
club_cmp = (cr7.team.str.replace(r" FC$", "", regex=True).value_counts().rename("csv")
            .to_frame().join(clubs["goles"].rename("excel")))
club_cmp["faltan_en_csv"] = club_cmp.excel - club_cmp.csv
club_cmp

,csv,excel,faltan_en_csv
team,,,
Real Madrid,450,450,0
Manchester United,145,145,0
Al-Nassr,115,130,15
Juventus,101,101,0
Sporting CP,5,5,0


El CSV de Ronaldo termina el **25-feb-2026** y le faltan 15 goles con Al-Nassr frente al Excel (más los 146 con Portugal, que nunca estuvieron). Conclusión: **para el total de Ronaldo manda el Excel**; el CSV sirve para análisis por competición/club.

## 4. Cruce por año calendario

Juntamos las tres fuentes en una sola tabla con `.join()` (une por índice = año).

In [16]:
cruce = (anual.set_index("year")[["messi", "ronaldo"]]
         .join(messi.groupby("anio").size().rename("messi_csv"))
         .join(cr7.groupby("anio").size().rename("cr7_csv_club"))
         .fillna(0).astype(int))

cruce["dif_messi"] = cruce.messi_csv - cruce.messi
cruce["cr7_no_en_csv"] = cruce.ronaldo - cruce.cr7_csv_club
cruce

,messi,ronaldo,messi_csv,cr7_csv_club,dif_messi,cr7_no_en_csv
year,,,,,,
2002,0,5,0,5,0,0
2003,0,1,0,1,0,0
2004,0,13,0,6,0,7
2005,3,15,3,13,0,2
2006,12,25,12,19,0,6
2007,31,34,31,29,0,5
2008,22,35,22,34,0,1
2009,41,30,41,29,0,1
2010,60,48,60,45,0,3


- **Messi**: Excel y CSV coinciden año por año; solo en 2026 el CSV tiene **8 goles más** → el CSV es más reciente.
- **Ronaldo**: la diferencia de cada año son sus goles con Portugal (y en 2026, además, los goles posteriores a febrero).

### ¿Hasta qué fecha llega el Excel?

El Excel no lo dice, pero el CSV de Messi numera cada gol (`Índice`). Si el Excel dice 922, buscamos el gol 922 y el 923:

In [17]:
messi.set_index("Índice").loc[[922, 923], ["fecha", "Equipo Local", "Equipo Visitante", "comp"]]

,fecha,Equipo Local,Equipo Visitante,comp
Índice,,,,
922,2026-08-19,Philadelphia Union,Inter Miami CF,Major League Soccer
923,2026-08-22,Inter Miami CF,Toronto FC,Major League Soccer


El Excel se cortó **entre el 19 y el 22 de agosto de 2026**. Ese es el "corte" del total de Ronaldo.

## 5. Cifra consolidada

In [18]:
resumen = pd.DataFrame({
    "goles": [len(messi), chequeo.loc["Ronaldo", "Overview"]],
    "fuente": ["messi_goles.csv", "Excel (Overview)"],
    "corte": [messi.fecha.max().date(), "~2026-08-20"],
}, index=["Messi", "Ronaldo"])
resumen["faltan_1000"] = 1000 - resumen.goles

# referencia externa: messivsronaldo.football (fuente original del Excel), actualizada al 17-oct-2026
resumen["ref_web"] = [932, 981]
resumen["desfase"] = resumen.ref_web - resumen.goles
resumen

,goles,fuente,corte,faltan_1000,ref_web,desfase
Messi,930,messi_goles.csv,2026-09-20,70,932,2
Ronaldo,977,Excel (Overview),~2026-08-20,23,981,4


In [19]:
for jug, row in resumen.iterrows():
    print(f"{jug:<8} {row.goles} goles (corte {row.corte}) -> le faltan {row.faltan_1000} para 1000")

Messi    930 goles (corte 2026-09-20) -> le faltan 70 para 1000
Ronaldo  977 goles (corte ~2026-08-20) -> le faltan 23 para 1000


### Formatos temporales (resumen para la fase 2)

| Fuente | Columna | Formato | Tipo tras limpiar |
|---|---|---|---|
| Excel | `Year` | `2025` | `int` |
| Excel | `Season` | `"2025/26"` (texto) | se queda como texto / se mapea al año de inicio |
| Excel | `Age` | `38` | `int` |
| messi_goles.csv | `Fecha` | `20-9-2026` / `22-01-2006` | `datetime64` |
| ronaldo csv | `match_date` | `07/10/02` | `datetime64` |
| ambos CSV | minuto | `"90+6"` / `"90'+5"` | texto (no lo necesitamos para el modelo) |

## 6. Guardar datos limpios para el modelo

In [20]:
out = RUTA / "data_limpia"
out.mkdir(exist_ok=True)

cols_m = ["fecha", "anio", "comp", "seleccion", "Equipo Local", "Equipo Visitante", "Minuto"]
messi[cols_m].to_csv(out / "messi_goles_limpio.csv", index=False)
cr7[["fecha", "anio", "competition", "team", "opponent", "goal_minute", "goal_method"]].to_csv(out / "cr7_goles_club_limpio.csv", index=False)
cruce.to_csv(out / "goles_por_anio.csv", index_label="anio")
resumen.to_csv(out/"resumen_totales.csv")

sorted(p.name for p in out.iterdir())

['cr7_goles_club_limpio.csv',
 'goles_por_anio.csv',
 'llegadas_bayes.csv',
 'messi_goles_limpio.csv',
 'modelo_base.csv',
 'parametros_modelo.json',
 'partidos.csv',
 'proyeccion_bayes.csv',
 'resultados_modelo_a.csv',
 'resultados_modelo_b.csv',
 'resumen_totales.csv']

### Partidos jugados (para medir eficiencia)

Los goles y los partidos salen de la **misma hoja y de la misma fecha de corte** (Overview del Excel, ~20-ago-2026). Si se mezclaran los 930 goles del CSV de Messi con los partidos del Excel, el ratio quedaría inflado: esos 8 goles extra se marcaron en partidos que el Excel todavía no cuenta.


In [21]:
partidos = pd.DataFrame({
    "jugador": ["Messi", "Ronaldo"],
    "partidos": [ov.loc["Total Appearances", "Lionel Messi"], ov.loc["Total Appearances", "Cristiano Ronaldo"]],
    "goles": [ov.loc["Total Goals", "Lionel Messi"], ov.loc["Total Goals", "Cristiano Ronaldo"]],
}).astype({"partidos": int, "goles": int})
partidos["goles_por_partido"] = partidos.goles / partidos.partidos
partidos["corte"] = "2026-08-20"

partidos.to_csv(out / "partidos.csv", index=False)
partidos.round(3)

,jugador,partidos,goles,goles_por_partido,corte
0,Messi,1168,922,0.789,2026-08-20
1,Ronaldo,1332,977,0.733,2026-08-20
